<a href="https://colab.research.google.com/github/sanatVibhor/Distributed-Multi-Agent-AI-Assistant/blob/sanat-working-branch/WW2RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===== RAG Pipeline: World War II Wikipedia Page (Phi-3-mini generation) =====
# Before running: Runtime -> Change runtime type -> T4 GPU

!pip install -q sentence-transformers chromadb transformers torch accelerate

import requests
import chromadb
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---------- 1. FETCH WIKIPEDIA PAGE (via MediaWiki extracts API) ----------
print("Fetching Wikipedia page...")
headers = {"User-Agent": "RAG-Project/1.0 (educational use; contact: your_email@example.com)"}
resp = requests.get(
    "https://en.wikipedia.org/w/api.php",
    params={
        "action": "query",
        "titles": "World War II",
        "prop": "extracts",
        "explaintext": True,
        "format": "json"
    },
    headers=headers
)
data = resp.json()
pages = data["query"]["pages"]
text = list(pages.values())[0]["extract"]
print(f"Fetched {len(text)} characters")

# ---------- 2. CHUNK THE TEXT ----------
def chunk_text(text, chunk_size=250, overlap=75):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(text)
print(f"Created {len(chunks)} chunks")

# ---------- 3. EMBED CHUNKS ----------
print("Loading embedding model...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(chunks, show_progress_bar=True)

# ---------- 4. STORE IN CHROMADB ----------
client = chromadb.Client()
collection = client.get_or_create_collection(name="wwii_wiki")
collection.add(
    documents=chunks,
    embeddings=embeddings.tolist(),
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)
print("Stored chunks in ChromaDB")

# ---------- 5. LOAD GENERATION MODEL (Phi-3-mini-4k-instruct) ----------
print("Loading generation model (Phi-3-mini)...")
phi_tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
phi_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

def generate_answer(prompt, max_length=300):
    messages = [{"role": "user", "content": prompt}]
    inputs = phi_tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True, return_dict=True).to(phi_model.device)
    outputs = phi_model.generate(**inputs, max_new_tokens=max_length, do_sample=False)
    response = phi_tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return response.strip()
# ---------- 6. RAG QUERY FUNCTION ----------
def ask(question, top_k=3):
    query_embedding = embed_model.encode([question]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=top_k)
    retrieved_chunks = results["documents"][0]
    context = "\n\n".join(retrieved_chunks)

    prompt = f"""Read the context carefully and answer ONLY the specific question asked. Do not include unrelated dates or facts.

Context:
{context}

Question: {question}

Instruction: Give a direct, specific answer to the question above, using only relevant information from the context.
Answer:"""

    answer = generate_answer(prompt)
    print(f"\nQ: {question}")
    print(f"A: {answer}")
    return answer

# ---------- 7. TEST IT ----------
ask("When did World War II start and end?")
ask("Which countries were the main Axis powers?")
ask("What caused World War II?")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 1.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sour

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

question_box = widgets.Text(
    placeholder="Ask something about World War II...",
    description="Question:",
    layout=widgets.Layout(width="600px")
)
ask_button = widgets.Button(description="Ask", button_style="primary")
output_area = widgets.Output()

def on_ask_clicked(b):
    with output_area:
        clear_output()
        question = question_box.value
        if not question.strip():
            print("Please type a question.")
            return
        print("Thinking...")
        answer = ask(question)
        clear_output()
        print(f"Q: {question}\n")
        print(f"A: {answer}")

ask_button.on_click(on_ask_clicked)

display(widgets.HBox([question_box, ask_button]))
display(output_area)